# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Here, we print out a list of all record sets, their IDs, and the fields they contain. Each record set's and field's unique `@id` can be used in later data extraction and manipulation steps.

In [ ]:
def print_record_sets(ds):
    print("Available record sets:")
    for record_set in ds.metadata.record_sets:
        print(f"\n- Record set name: {record_set.name}")
        print(f"  @id: {record_set.id}")
        if hasattr(record_set, 'fields') and record_set.fields:
            print("  Fields:")
            for field in record_set.fields:
                print(f"    - {field.name}, @id: {field.id}, dtype: {field.data_type}")

print_record_sets(dataset)

## 3. Data Extraction
Load data from one or more record sets into a DataFrame for analysis.

Replace the record set and field `@id`s in the examples below with those discovered in the previous section.

We'll enumerate the available record sets, then extract data for one as an example.

In [ ]:
# Extract all available record set @ids
record_sets = [r.id for r in dataset.metadata.record_sets]
print("Record set IDs:", record_sets)

dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set '{record_set_id}' with shape {df.shape} and columns: {df.columns.tolist()}")

# For demonstration, pick the first available record set with data:
example_record_set_id = next((r for r in record_sets if r in dataframes), None)
if example_record_set_id:
    print(f"\nExample data from record set {example_record_set_id}:")
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's select a numeric field (column) from the record set DataFrame for analysis. Update `numeric_field_id` and `group_field_id` as appropriate from your record set.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# You may need to set these IDs based on actual field names from your overview section
# Example (modeling after typical protein MS datasets):
# Let's guess some likely numeric columns: 'molecular_weight', 'coverage', 'peptide_count', etc.
# Update these as per your dataset field list output above!
df = dataframes[example_record_set_id]

possible_numeric_fields = [c for c in df.columns if df[c].dtype in [np.float64, np.int64, np.float32, np.int32]]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    # Try to coerce a typical column name to numeric
    for col in df.columns:
        if any(x in col.lower() for x in ['molecular', 'weight', 'coverage', 'peptide', 'mw', 'count', 'abundance']):
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                numeric_field_id = col
                break
            except Exception:
                continue
    else:
        numeric_field_id = df.columns[0]  # fallback

print(f"Selected numeric field: {numeric_field_id}")

# Filter: Remove missing values and filter records where numeric_field > threshold (using 10 or 100 as threshold)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}: {filtered_df.shape[0]} rows")

# Normalize field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to find a categorical field for grouping (e.g., 'description', 'id', or anything with <20 unique values)
group_field_candidates = [c for c in df.columns if df[c].nunique() < 20 and c != numeric_field_id]
group_field = group_field_candidates[0] if group_field_candidates else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
    print(f"\nGrouped data by '{group_field}' (mean {numeric_field_id}):")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between numeric fields in the dataset.

Example: Distribution of the selected numeric field, or scatter plot between two numeric fields if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f'Distribution of {numeric_field_id} (filtered > {threshold})')
plt.show()

# If two numeric columns, show scatter plot
if len(possible_numeric_fields) > 1:
    plt.figure(figsize=(6,5))
    sns.scatterplot(data=filtered_df, x=possible_numeric_fields[0], y=possible_numeric_fields[1])
    plt.xlabel(possible_numeric_fields[0])
    plt.ylabel(possible_numeric_fields[1])
    plt.title(f'Scatter: {possible_numeric_fields[0]} vs {possible_numeric_fields[1]}')
    plt.show()

## 6. Conclusion
In this notebook, we explored the dataset **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** using the `mlcroissant` library. We loaded available record sets, extracted data frames, performed filtering and normalization on numeric fields, and visualized underlying distributions.

You can extend this analysis by examining other fields or record sets, joining data from multiple record sets, or applying advanced machine learning workflows.